# Proiectul 2 · Varianta 2: Piatră, hârtie sau foarfecă?
### Clasificare de imagini cu transfer learning (ResNet18)

**Autor:** Georgiana Stan

---

**Rezumat.** În această variantă aplic același proces ca în Varianta 1 pe un alt set de date: poze cu mâini care fac gesturile din jocul piatră-hârtie-foarfecă, deci **3 clase**. Pornesc tot de la ResNet18 pre-antrenat pe ImageNet și compar feature extraction cu fine-tuning. Codul este scris astfel încât să poată fi folosit și pentru alte seturi de poze: se schimbă doar celula de setări.

### Cuprins
| # | Secțiune | Conținut |
|---|-----|-----------|
| 1 | Problema și datele | 3 clase, sursa pozelor |
| 2 | Setări | toate setările proiectului, într-o singură celulă |
| – | Pregătire | biblioteci și funcții ajutătoare |
| 3 | Datele | obținere, împărțire train/val, verificare, explorare, augmentare |
| 4 | Baseline | „ghicesc mereu clasa mare” |
| 5 | Modele | feature extraction vs. fine-tuning |
| 6 | Evaluare | matrice de confuzie, precizie/recall, greșeli |
| 7 | Demo | salvez modelul, test pe poze noi |
| 8 | Concluzii | rezultate, limitări, pași următori |

## 1 · Problema și datele

- **Întrebarea:** primind o poză cu o mână, ce gest face: **piatră** (`rock`), **hârtie** (`paper`) sau **foarfecă** (`scissors`)?
- **Tipul problemei:** clasificare de imagini, cu 3 clase echilibrate.
- **Datele:** setul *Rock Paper Scissors* creat de Laurence Moroney (licență CC-BY 2.0). Conține mâini generate pe calculator (CGI), cu diferite culori ale pielii, pe fundal alb, la 300×300 px. Setul vine deja împărțit: `rps` (aproximativ 2.500 de poze) pentru antrenare și `rps-test-set` (aproximativ 370 de poze, cu alte mâini), pe care îl folosesc pentru validare.
- **La ce ar folosi:** un joc piatră-hârtie-foarfecă cu camera web, contra calculatorului.
- **Metrica:** acuratețea pe validare, plus matricea de confuzie, ca să văd ce gesturi se confundă.
- **Ce înseamnă „bine”:** cu 3 clase egale, baseline-ul este de aproximativ 33%. Ținta mea este peste 90%.
- **O întrebare în plus:** un model antrenat pe mâini desenate pe calculator recunoaște și o mână reală, fotografiată cu telefonul? (secțiunea 7)

## 2 · Setări
Toate setările proiectului sunt într-o singură celulă. Codul poate lua pozele din mai multe surse (`SURSA`):

| `SURSA` | De unde iau pozele |
|---|---|
| `"rps"` | setul piatră-hârtie-foarfecă (folosit în acest proiect) |
| `"zip"` | o arhivă ZIP (`CALE_ZIP`) |
| `"drive"` | o arhivă sau un folder din Google Drive |
| `"folder"` | un folder local (`CALE_FOLDER`) |
| `"url"` | o arhivă descărcată de la un link (`URL_ARHIVA`) |
| `"kaggle"` | un set de date de pe Kaggle (`KAGGLE_SET`) |

Pozele pot fi organizate fie câte un folder pentru fiecare clasă (atunci le împart automat 80% train / 20% val), fie deja în `train/` și `val/`.

Celelalte setări:
- `CLASE_ALESE`: dacă vreau să folosesc doar o parte din clase;
- `MAX_POZE_PE_CLASA`: limitează numărul de poze pe clasă. Setul are aproximativ 840 de poze pe clasă; eu folosesc 150 (120 pentru antrenare și 30 pentru validare), ca antrenarea să fie rapidă;
- `ARHITECTURA`: modelul pre-antrenat (`resnet18`);
- `STRATEGII`: strategiile pe care le compar;
- `EPOCI`: de câte ori trece modelul prin toate pozele de antrenare; `BATCH`: câte poze sunt într-un lot.

In [ ]:
import os   # am nevoie de os aici, pentru ultima linie
# ======================= SETĂRI PROIECT =======================
NUME_PROIECT = "piatra_hartie_foarfeca"   # apare în numele fișierelor salvate

SURSA = "rps"                             # sursa folosită în proiect; alte opțiuni: "zip" | "drive" | "folder" | "url" | "kaggle"

# Variabilele de mai jos se folosesc DOAR dacă SURSA are altă valoare decât "rps":
CALE_ZIP = "/content/pozele_mele.zip"     # pentru "zip" / "drive"
CALE_FOLDER = "/content/pozele_mele"      # pentru "folder" (sau "drive" fără zip)
URL_ARHIVA = "https://storage.googleapis.com/download.tensorflow.org/data/rps.zip"   # pentru "url" (împărțire automată 80/20)
KAGGLE_SET = ""                           # pentru "kaggle": "utilizator/nume-set"

CLASE_ALESE = None            # None = toate clasele găsite; sau ex. ["rock", "paper"]
MAX_POZE_PE_CLASA = 150       # None = toate; un număr mic face antrenarea mai rapidă
PROCENT_VAL = 0.2             # 20% din poze merg la validare

ARHITECTURA = "resnet18"      # "resnet18" | "resnet50" | "efficientnet_b0" | "mobilenet_v3_large"
STRATEGII = ["feature_extraction", "fine_tuning"]   # le compară pe toate din listă
EPOCI = 5
BATCH = 16
# ==============================================================

# Unde salvez pozele pregătite (train/ + val/): în /content dacă există, altfel lângă notebook
CALE_DATE = f"/content/{NUME_PROIECT}_date" if os.path.isdir("/content") else f"{NUME_PROIECT}_date"
print("Pozele pregătite vor fi în:", CALE_DATE)

## Pregătire: biblioteci și funcții ajutătoare
În această parte instalez bibliotecile care lipsesc și definesc funcțiile folosite în restul proiectului (aceleași ca în Varianta 1). Aici nu se antrenează încă niciun model.

### Biblioteci folosite

| Bibliotecă | La ce o folosesc |
|---|---|
| `torch` (PyTorch) | rețele neuronale: modele, antrenare, calcule pe GPU |
| `torchvision` | modele pre-antrenate (ResNet18), încărcarea pozelor din foldere, transformări pentru imagini |
| `numpy` | calcule cu vectori și matrici |
| `pandas` | tabele (numărul de poze, rezultate) |
| `matplotlib` | grafice și afișarea pozelor |
| `scikit-learn` | metrici de evaluare: matricea de confuzie, precizie, recall |
| `pillow` (PIL) | deschiderea fișierelor imagine (JPG, PNG) |
| `requests` | descărcarea fișierelor de pe internet |

Celula următoare verifică ce biblioteci lipsesc și le instalează doar pe acelea, cu `pip`. Dacă sunt toate instalate, afișează doar versiunile lor.

In [ ]:
import importlib.util, importlib.metadata, subprocess, sys

# numele cu care IMPORTĂM biblioteca  ->  numele cu care o INSTALĂM cu pip (uneori diferă!)
NECESARE = {
    "torch": "torch",
    "torchvision": "torchvision",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "PIL": "pillow",
    "requests": "requests",
}

def versiune(pachet):
    """Versiunea instalată a unui pachet, ca tuplu (major, minor). (0, 0) dacă nu e instalat."""
    try:
        return tuple(int(x) for x in importlib.metadata.version(pachet).split("+")[0].split(".")[:2])
    except Exception:
        return (0, 0)

# find_spec(nume) întoarce None dacă biblioteca NU e instalată
lipsa = [pip for modul, pip in NECESARE.items() if importlib.util.find_spec(modul) is None]

# Avem nevoie de torchvision >= 0.15 (acolo a apărut models.get_model, pe care îl folosim)
if "torchvision" not in lipsa and versiune("torchvision") < (0, 15):
    lipsa.append("torchvision>=0.15")

if lipsa:
    print("📦 Instalez:", ", ".join(lipsa), "(poate dura câteva minute)")
    # echivalentul comenzii din terminal:  pip install -q --upgrade torch torchvision ...
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *lipsa])
    print("✅ Instalare terminată. Dacă apar erori la importuri, repornesc kernel-ul și rulez din nou.")
else:
    print("✅ Toate bibliotecile sunt deja instalate.")

# Afișăm versiunile (util dacă ceva nu merge și ceri ajutor)
for pachet in ["torch", "torchvision", "scikit-learn", "pandas"]:
    print(f"   {pachet:<13} {importlib.metadata.version(pachet)}")

### Importuri, seed și device

- **Seed-ul** fixează punctul de pornire al numerelor aleatoare. De el depind amestecarea pozelor, augmentarea și greutățile inițiale ale stratului nou. Cu același seed (42), rezultatele ies aproape la fel la fiecare rulare, așa că pot compara corect experimentele între ele.
- **Device-ul** este locul unde se fac calculele: placa video (`cuda`, mult mai rapidă) sau procesorul (`cpu`).

In [ ]:
# --- biblioteci standard Python ---
import os, io, random, zipfile, tarfile, hashlib, shutil, time   # fișiere, arhive, amprente, timp

# --- biblioteci pentru date și grafice ---
import numpy as np                    # calcule cu vectori
import pandas as pd                   # tabele
import matplotlib.pyplot as plt       # grafice
import requests                       # descărcări de pe internet
from PIL import Image                 # deschiderea pozelor

# --- PyTorch ---
import torch
import torch.nn as nn                                       # straturi (ex. nn.Linear) și funcții de loss
from torchvision import datasets, transforms, models        # seturi de poze, transformări, modele pre-antrenate

# --- metrici de evaluare ---
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

plt.rcParams["figure.figsize"] = (8, 4.5)                   # mărimea implicită a graficelor

SEED = 42
def fixeaza_seed(seed=SEED):
    """Fixează toate generatoarele de numere aleatoare (Python, NumPy, PyTorch)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
fixeaza_seed()

# Folosim GPU-ul dacă există, altfel procesorul
device = "cuda" if torch.cuda.is_available() else "cpu"
# NUM_WORKERS = câte procese încarcă pozele în paralel cu antrenarea (pe GPU ajută, pe CPU nu)
NUM_WORKERS = 2 if device == "cuda" else 0
print("Rulăm pe:", device, "| PyTorch", torch.__version__)
if device == "cpu":
    print("⚠️ Nu există GPU: antrenarea pe CPU va fi mai lentă.")

### Funcții pentru antrenare și evaluare

`antreneaza` și `evalueaza` au același rol ca `.fit()` și `.score()` din scikit-learn.

**Cum învață rețeaua**, la fiecare pas:
1. primește un **lot (batch)** de 16 poze și calculează un scor pentru fiecare clasă (**forward**);
2. **funcția de loss** (`CrossEntropyLoss`) măsoară cât de departe sunt scorurile de etichetele corecte;
3. **backward** calculează **gradienții**: în ce direcție trebuie schimbată fiecare greutate ca loss-ul să scadă;
4. **optimizatorul** `Adam` modifică greutățile cu un pas mic. Mărimea pasului este **rata de învățare (`lr`)**.

O trecere prin toate pozele de antrenare se numește **epocă**. După fiecare epocă calculez **acuratețea pe validare**, adică pe poze pe care modelul nu le-a folosit la antrenare. Ea arată cât de bine **generalizează** modelul. Dacă acuratețea pe train ajunge mult peste cea pe validare, modelul a început să memoreze pozele de antrenare (**overfitting**).

In [ ]:
# Media și deviația standard ale culorilor (R, G, B) din ImageNet.
# Modelele pre-antrenate au văzut pozele normalizate cu aceste valori, deci le normalizăm la fel.
MEDIE, DEVIATIE = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def antreneaza(model, train_loader, epoci=1, lr=1e-3, val_loader=None):
    """Antrenează modelul și întoarce (model, istoric). Istoricul = tabel cu loss/acuratețe pe fiecare epocă."""
    model.to(device)                                              # mutăm modelul pe GPU/CPU
    # Optimizăm DOAR parametrii neînghețați (requires_grad=True)
    parametri = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(parametri, lr=lr)
    loss_fn = nn.CrossEntropyLoss()                               # loss-ul standard pentru clasificare
    istoric = []
    for epoca in range(1, epoci + 1):
        start = time.time()
        model.train()                                             # modul „antrenare” (contează pentru BatchNorm/Dropout)
        total = corecte = 0
        suma_loss = 0.0
        for x, y in train_loader:                                 # x = lot de poze [16, 3, 224, 224], y = etichete [16]
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()                                 # ștergem gradienții de la lotul anterior
            iesire = model(x)                                     # forward: un scor pentru fiecare clasă
            loss = loss_fn(iesire, y)                             # cât de greșit e modelul pe acest lot
            loss.backward()                                       # backward: calculăm gradienții
            optimizer.step()                                      # actualizăm greutățile
            suma_loss += loss.item() * len(y)
            corecte += (iesire.argmax(1) == y).sum().item()       # argmax = clasa cu scorul cel mai mare
            total += len(y)
        rand = {"epoca": epoca, "loss": suma_loss / total, "acc_train": corecte / total}
        mesaj = f"epoca {epoca}: loss {rand['loss']:.3f} | acuratețe train {rand['acc_train']:.3f}"
        if val_loader is not None:
            rand["acc_val"] = evalueaza(model, val_loader)[0]
            mesaj += f" | acuratețe validare {rand['acc_val']:.3f}"
        print(mesaj + f"  ({time.time() - start:.0f}s)")
        istoric.append(rand)
    return model, pd.DataFrame(istoric)

@torch.no_grad()                    # la evaluare nu avem nevoie de gradienți: mai rapid, mai puțină memorie
def evalueaza(model, loader):
    """Întoarce (acuratețe, etichete reale, predicții, probabilități pentru fiecare clasă)."""
    model.to(device).eval()         # modul „evaluare”: fără Dropout, BatchNorm folosește statisticile învățate
    y_true, y_pred, probs = [], [], []
    for x, y in loader:
        p = model(x.to(device)).softmax(1).cpu()       # softmax transformă scorurile în probabilități (sumă = 100%)
        probs.append(p)
        y_pred += p.argmax(1).tolist()
        y_true += y.tolist()
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return (y_true == y_pred).mean(), y_true, y_pred, torch.cat(probs).numpy()

def deseneaza_istoric(istorice):
    """Desenează curbele de învățare. istorice = {"nume experiment": tabelul întors de antreneaza}."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    for nume, ist in istorice.items():
        ax1.plot(ist["epoca"], ist["loss"], marker="o", label=nume)
        linie, = ax2.plot(ist["epoca"], ist["acc_val"], marker="o", label=f"{nume} · validare")
        ax2.plot(ist["epoca"], ist["acc_train"], ls="--", color=linie.get_color(), alpha=0.6,
                 label=f"{nume} · train")
    ax1.set(title="Loss pe train (mai mic = mai bine)", xlabel="epoca"); ax1.legend()
    ax2.set(title="Acuratețe (plin = validare, punctat = train)", xlabel="epoca", ylim=(0, 1.02))
    ax2.legend(fontsize=8)
    for ax in (ax1, ax2):
        ax.grid(alpha=0.3)
        ax.xaxis.get_major_locator().set_params(integer=True)   # pe axa epocilor doar numere întregi
    plt.show()

def incarca_imagine(sursa):
    """Deschide o imagine dintr-un URL (http...) sau dintr-un fișier local și o aduce la RGB (3 canale)."""
    if str(sursa).startswith("http"):
        raspuns = requests.get(sursa, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
        raspuns.raise_for_status()                             # oprește cu eroare dacă descărcarea a eșuat
        return Image.open(io.BytesIO(raspuns.content)).convert("RGB")
    return Image.open(sursa).convert("RGB")

def arata_tensor(x, ax, titlu=""):
    """Afișează un tensor normalizat [3, H, W] ca imagine (anulăm normalizarea ca să arate normal)."""
    img = x.permute(1, 2, 0).numpy() * DEVIATIE + MEDIE      # [3,H,W] -> [H,W,3] și de-normalizare
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(titlu, fontsize=9)
    ax.axis("off")

def numara_parametri(model):
    """Câte numere (greutăți) învață modelul, din câte are în total."""
    antrenabili = sum(p.numel() for p in model.parameters() if p.requires_grad)
    toti = sum(p.numel() for p in model.parameters())
    print(f"parametri antrenați: {antrenabili:,} din {toti:,} ({antrenabili / toti:.2%})")
    return antrenabili

### Funcții pentru verificarea datelor

- `numara_poze`: un tabel cu numărul de poze din fiecare clasă, în train și în val.
- `verifica_poze`: deschide fiecare poză; cele stricate sunt mutate deoparte, pentru că ar opri antrenarea cu o eroare.
- `gaseste_duplicate`: caută poze identice în train și în val. Pentru fiecare fișier calculez o „amprentă” (**hash MD5**); două fișiere identice au aceeași amprentă. O poză de validare pe care modelul a văzut-o deja la antrenare ar face acuratețea să pară mai mare decât e.
- `statistici_dimensiuni` și `arata_mostre`: mărimea pozelor și câteva exemple din fiecare clasă.

In [ ]:
# Formatele de imagine pe care le acceptăm (aceleași pe care le citește ImageFolder)
EXTENSII = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")

def poze_din(folder):
    """Lista (sortată) cu fișierele imagine dintr-un folder."""
    return sorted(f for f in os.listdir(folder) if f.lower().endswith(EXTENSII))

def numara_poze(cale_date):
    """Tabel: rânduri = clase, coloane = train / val, valori = numărul de poze."""
    randuri = []
    for impartire in ["train", "val"]:
        for clasa in sorted(os.listdir(f"{cale_date}/{impartire}")):
            if os.path.isdir(f"{cale_date}/{impartire}/{clasa}"):
                randuri.append({"impartire": impartire, "clasa": clasa,
                                "poze": len(poze_din(f"{cale_date}/{impartire}/{clasa}"))})
    return pd.DataFrame(randuri).pivot(index="clasa", columns="impartire", values="poze")

def verifica_poze(cale_date):
    """Încearcă să deschidă fiecare poză. Pozele stricate sunt mutate în <cale_date>_stricate/."""
    stricate = []
    for radacina, _, fisiere in os.walk(cale_date):           # parcurgem toate subfolderele
        for f in fisiere:
            if not f.lower().endswith(EXTENSII):
                continue
            cale = os.path.join(radacina, f)
            try:
                with Image.open(cale) as img:
                    img.convert("RGB").load()                 # .load() citește efectiv pixelii
            except Exception as e:
                stricate.append(cale)
                destinatie = cale.replace(cale_date, cale_date + "_stricate", 1)
                os.makedirs(os.path.dirname(destinatie), exist_ok=True)
                shutil.move(cale, destinatie)
                print("❌ mutată (nu se deschide):", cale, "|", type(e).__name__)
    print(f"Am verificat pozele: {len(stricate)} stricate.")
    return stricate

def hash_fisier(cale):
    """„Amprenta” MD5 a unui fișier: două fișiere identice au aceeași amprentă."""
    with open(cale, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def gaseste_duplicate(cale_date):
    """Caută aceeași poză (identică, byte cu byte) și în train, și în val."""
    hash_train = {}                                           # amprentă -> fișier din train
    for clasa in os.listdir(f"{cale_date}/train"):
        for f in poze_din(f"{cale_date}/train/{clasa}"):
            hash_train[hash_fisier(f"{cale_date}/train/{clasa}/{f}")] = f"train/{clasa}/{f}"
    duplicate = []
    for clasa in os.listdir(f"{cale_date}/val"):
        for f in poze_din(f"{cale_date}/val/{clasa}"):
            h = hash_fisier(f"{cale_date}/val/{clasa}/{f}")
            if h in hash_train:                               # aceeași amprentă există și în train
                duplicate.append((hash_train[h], f"val/{clasa}/{f}"))
    print(f"Poze identice în train ȘI în val: {len(duplicate)}")
    for a, b in duplicate[:10]:
        print("   ", a, "==", b)
    return duplicate

def statistici_dimensiuni(cale_date, impartire="train"):
    """Tabel cu lățimea și înălțimea fiecărei poze."""
    randuri = []
    for clasa in sorted(os.listdir(f"{cale_date}/{impartire}")):
        for f in poze_din(f"{cale_date}/{impartire}/{clasa}"):
            with Image.open(f"{cale_date}/{impartire}/{clasa}/{f}") as img:
                randuri.append({"clasa": clasa, "latime": img.size[0], "inaltime": img.size[1]})
    return pd.DataFrame(randuri)

def arata_mostre(cale_date, clase, n=6, impartire="train"):
    """Afișează n poze alese la întâmplare din fiecare clasă (un rând pe clasă)."""
    fig, axe = plt.subplots(len(clase), n, figsize=(2.5 * n, 2.6 * len(clase)), squeeze=False)
    for rand, clasa in enumerate(clase):
        fisiere = poze_din(f"{cale_date}/{impartire}/{clasa}")
        for ax, f in zip(axe[rand], random.Random(SEED).sample(fisiere, min(n, len(fisiere)))):
            img = Image.open(f"{cale_date}/{impartire}/{clasa}/{f}")
            ax.imshow(img)
            ax.set_title(f"{clasa} · {img.size[0]}x{img.size[1]}", fontsize=9)
        for ax in axe[rand]:
            ax.axis("off")
    plt.tight_layout()
    plt.show()

### Funcții pentru model: transfer learning

**Ideea transfer learning-ului:** ResNet18 a fost antrenat pe **ImageNet** (1,2 milioane de poze, 1000 de clase) și a învățat deja să recunoască margini, texturi, forme și părți de obiecte. Nu îl mai antrenez de la zero. Păstrez ce știe și înlocuiesc doar **ultimul strat** (`fc`): în loc de 1000 de ieșiri, noul strat are câte o ieșire pentru fiecare clasă a mea.

```text
poza → [layer1 → layer2 → layer3 → layer4]  →  fc nou  → scor pentru fiecare clasă
        extractorul de caracteristici            (512 → nr. clase)
        (pre-antrenat pe ImageNet)
```

**Strategii** (cât las modelul să se schimbe):
- **feature extraction**: îngheț tot modelul (`requires_grad = False`) și antrenez doar stratul nou. Puține poze → puțini parametri de învățat → risc mic de overfitting.
- **fine-tuning**: dezgheț și **ultimul bloc** (`layer4`) și folosesc un `lr` de 10 ori mai mic, ca să nu stric ce a învățat modelul pe ImageNet.
- **fine-tuning complet**: se antrenează tot modelul, cu un `lr` foarte mic. Are sens doar cu multe poze sau cu poze foarte diferite de cele din ImageNet.

**Transformări:** toate pozele trebuie aduse la formatul cerut de model (`[3, 224, 224]`, normalizate cu media și deviația standard din ImageNet). La **train** adaug **augmentare**: la fiecare epocă, poza este modificată puțin, la întâmplare, așa că modelul vede mereu variante noi, ca și cum aș avea mai multe date. La **validare** nu modific nimic. Greutățile pre-antrenate (aproximativ 45 MB) se descarcă automat la prima folosire.

In [ ]:
def model_adaptat(nume, nr_clase, strategie):
    """Model pre-antrenat pe ImageNet + cap nou cu `nr_clase` ieșiri.

    strategie:
      "feature_extraction"   -> înghețat tot, se antrenează doar capul nou
      "fine_tuning"          -> se antrenează și ultimul bloc (layer4 la ResNet)
      "fine_tuning_complet"  -> se antrenează tot modelul (lr foarte mic!)
    """
    greutati = models.get_model_weights(nume).DEFAULT         # cele mai bune greutăți ImageNet disponibile
    model = models.get_model(nume, weights=greutati)          # se descarcă automat prima dată

    # ❄️ înghețăm tot (sau nimic, la fine-tuning complet)
    for p in model.parameters():
        p.requires_grad = (strategie == "fine_tuning_complet")

    # 🔥 la fine-tuning dezghețăm ultimul bloc
    if strategie == "fine_tuning":
        ultimul_bloc = model.layer4 if hasattr(model, "layer4") else model.features[-1]
        for p in ultimul_bloc.parameters():
            p.requires_grad = True

    # Capul nou: un strat nn.Linear(câte numere intră, câte clase avem).
    # Un strat nou creat are requires_grad=True, deci se antrenează mereu.
    if hasattr(model, "fc"):                                  # ResNet: ultimul strat se numește fc
        model.fc = nn.Linear(model.fc.in_features, nr_clase)
    else:                                                     # EfficientNet / MobileNet: classifier[-1]
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, nr_clase)
    return model, greutati.transforms()

def transformari_pentru(nume):
    """Întoarce (transform_train cu augmentare, transform_val fără augmentare) potrivite modelului."""
    # transform_val = preprocesarea oficială a modelului: Resize(256) -> CenterCrop(224) -> tensor -> normalizare
    transform_val = models.get_model_weights(nume).DEFAULT.transforms()
    marime = transform_val.crop_size[0]                                    # 224 la resnet18
    transform_train = transforms.Compose([
        transforms.RandomResizedCrop(marime, scale=(0.6, 1.0)),   # o bucată aleatoare (60–100% din poză), adusă la 224x224
        transforms.RandomHorizontalFlip(),                        # oglindire stânga-dreapta, 50% din timp
        transforms.ColorJitter(brightness=0.2, contrast=0.2),     # puțin mai luminos/întunecat, mai mult/mai puțin contrast
        transforms.ToTensor(),                                    # imagine -> tensor [3, 224, 224] cu valori 0..1
        transforms.Normalize(transform_val.mean, transform_val.std),   # normalizarea ImageNet
    ])
    return transform_train, transform_val

# Rata de învățare potrivită pentru fiecare strategie (mai mulți parametri dezghețați -> pași mai mici)
LR_IMPLICIT = {"feature_extraction": 1e-3, "fine_tuning": 1e-4, "fine_tuning_complet": 1e-5}

def ruleaza_experiment(nume_arhitectura, strategie, epoci, train_loader, val_loader, nr_clase, lr=None):
    """Construiește modelul, îl antrenează și întoarce (model, istoric, acuratețe validare)."""
    fixeaza_seed()                                    # fiecare experiment pornește din același punct: comparație cinstită
    model, _ = model_adaptat(nume_arhitectura, nr_clase, strategie)
    numara_parametri(model)
    lr = lr or LR_IMPLICIT[strategie]
    start = time.time()
    model, istoric = antreneaza(model, train_loader, epoci=epoci, lr=lr, val_loader=val_loader)
    acc = evalueaza(model, val_loader)[0]
    print(f"→ {nume_arhitectura} · {strategie}: acuratețe validare {acc:.1%} în {time.time() - start:.0f}s\n")
    return model, istoric, acc

### Funcții pentru evaluare, salvare și predicție

- `matrice_confuzie`: un tabel cu clasa reală pe rânduri și clasa prezisă pe coloane. Pe diagonală sunt răspunsurile corecte; în rest se vede ce clase se confundă.
- `arata_greselile`: pozele greșite, începând cu cele la care modelul era cel mai sigur pe el.
- `arata_nesigure`: pozele la care modelul a ezitat cel mai mult (probabilități apropiate între clase).
- `salveaza_model` / `incarca_model`: salvez greutățile, lista claselor și numele arhitecturii. La încărcare reconstruiesc arhitectura și pun peste ea greutățile salvate.
- `prezice`: clasifică o poză nouă (fișier sau link). Poza trece prin aceeași preprocesare ca pozele de validare, fără augmentare.

In [ ]:
def matrice_confuzie(y_true, y_pred, clase, titlu=""):
    """Matricea de confuzie: rânduri = realitatea, coloane = ce a prezis modelul."""
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=range(len(clase)),
                                            display_labels=clase, cmap="Blues", colorbar=False)
    plt.xlabel("ce spune modelul"); plt.ylabel("realitate"); plt.title(titlu)
    plt.show()

def arata_greselile(dataset_val, y_true, y_pred, probs, clase, max_poze=8):
    """Afișează pozele clasificate greșit, cu probabilitatea dată de model."""
    gresite = np.where(y_true != y_pred)[0]                              # pozițiile pozelor greșite
    print(f"Greșeli: {len(gresite)} din {len(y_true)}")
    if len(gresite) == 0:
        return []
    gresite = gresite[np.argsort(-probs[gresite].max(1))][:max_poze]    # sortăm: întâi cele „sigure pe ele”
    fig, axe = plt.subplots(1, len(gresite), figsize=(3 * len(gresite), 3.4), squeeze=False)
    for ax, i in zip(axe[0], gresite):
        arata_tensor(dataset_val[i][0], ax,
                     f"real: {clase[y_true[i]]}\nprezis: {clase[y_pred[i]]} ({probs[i].max():.0%})")
    plt.tight_layout()
    plt.show()
    return [dataset_val.imgs[i][0] for i in gresite]                     # căile fișierelor greșite

def arata_nesigure(dataset_val, y_true, probs, clase, max_poze=6):
    """Pozele la care modelul a ezitat cel mai mult (probabilitatea maximă cea mai mică)."""
    idx = np.argsort(probs.max(1))[:max_poze]
    fig, axe = plt.subplots(1, len(idx), figsize=(3 * len(idx), 3.4), squeeze=False)
    for ax, i in zip(axe[0], idx):
        arata_tensor(dataset_val[i][0], ax,
                     f"real: {clase[y_true[i]]}\n" + " | ".join(f"{c} {p:.0%}" for c, p in zip(clase, probs[i])))
    plt.tight_layout()
    plt.show()

def salveaza_model(model, clase, arhitectura, cale):
    """Salvează greutățile (state_dict), lista claselor și numele arhitecturii într-un fișier .pt."""
    torch.save({"greutati": model.state_dict(), "clase": clase, "arhitectura": arhitectura}, cale)
    print(f"Salvat: {cale} ({os.path.getsize(cale) / 1e6:.1f} MB)")

def incarca_model(cale):
    """Reconstruiește arhitectura, pune capul cu nr. corect de clase și încarcă greutățile salvate."""
    salvat = torch.load(cale, map_location=device)
    arhitectura = salvat.get("arhitectura", "resnet18")      # modelul din curs nu avea cheia asta -> resnet18
    model = models.get_model(arhitectura, weights=None)      # arhitectura „goală” (greutăți aleatoare)...
    if hasattr(model, "fc"):
        model.fc = nn.Linear(model.fc.in_features, len(salvat["clase"]))
    else:
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(salvat["clase"]))
    model.load_state_dict(salvat["greutati"])                # ...peste care punem greutățile antrenate
    return model.to(device).eval(), salvat["clase"], arhitectura

@torch.no_grad()
def prezice(sursa, model, clase, transform, arata=True):
    """Clasifică o poză nouă. Întoarce (clasa prezisă, probabilitatea ei)."""
    img = incarca_imagine(sursa)
    x = transform(img).unsqueeze(0).to(device)               # unsqueeze(0): [3,224,224] -> [1,3,224,224] (un lot cu o poză)
    p = model.eval()(x).softmax(1)[0].cpu()                  # probabilitățile pentru fiecare clasă
    if arata:
        plt.figure(figsize=(4, 4))
        plt.imshow(img); plt.axis("off")
        plt.title(" | ".join(f"{c}: {pi:.0%}" for c, pi in zip(clase, p)), fontsize=10)
        plt.show()
    return clase[int(p.argmax())], float(p.max())

## 3 · Datele
### 3.1 Obținerea pozelor
Pozele ajung într-un folder „brut” (`..._brut`), exact cum au fost descărcate. Pentru setul piatră-hârtie-foarfecă descarc două arhive: `rps.zip` (antrenare) și `rps-test-set.zip` (validare). Validarea conține mâini pe care modelul nu le vede la antrenare, deci este o verificare mai corectă decât o împărțire la întâmplare. Pentru fiecare arhivă am pus mai multe linkuri, în caz că unul nu funcționează.

Funcția `gaseste_radacina` coboară prin foldere până găsește fie `train/` și `val/`, fie folderele claselor, pentru că arhivele au deseori un folder în plus.

In [ ]:
DIR_BRUT = CALE_DATE + "_brut"            # aici ajung pozele exact cum au venit (nepregătite)

def dezarhiveaza(cale_arhiva, destinatie):
    """Dezarhivează un .zip sau .tgz/.tar.gz în folderul `destinatie`."""
    os.makedirs(destinatie, exist_ok=True)
    if cale_arhiva.endswith(".zip"):
        zipfile.ZipFile(cale_arhiva).extractall(destinatie)
    else:                                                    # .tgz / .tar.gz
        with tarfile.open(cale_arhiva) as t:
            try:
                t.extractall(destinatie, filter="data")
            except TypeError:                                # Python mai vechi, fără `filter`
                t.extractall(destinatie)
    print("📦 dezarhivat în", destinatie)

URL_RPS = {   # mai multe oglinzi, în caz că una nu merge
    "rps.zip": ["https://storage.googleapis.com/download.tensorflow.org/data/rps.zip",
                "https://storage.googleapis.com/tensorflow-1-public/course2/week4/rps.zip",
                "https://storage.googleapis.com/learning-datasets/rps.zip"],
    "rps-test-set.zip": ["https://storage.googleapis.com/download.tensorflow.org/data/rps-test-set.zip",
                         "https://storage.googleapis.com/tensorflow-1-public/course2/week4/rps-test-set.zip",
                         "https://storage.googleapis.com/learning-datasets/rps-test-set.zip"],
}

def descarca(url, nume):
    """Descarcă fișierul de la `url` și îl salvează cu numele `nume` (pe bucăți, ca să nu umple memoria)."""
    print("⬇️ descarc", url)
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(nume, "wb") as f:
            shutil.copyfileobj(r.raw, f)

def adu_rps():
    """rps.zip → train/, rps-test-set.zip → val/ (setul vine deja împărțit)."""
    for nume, linkuri in URL_RPS.items():
        if os.path.exists(nume):                       # ex. arhivă pusă manual lângă notebook
            print("✅ am deja", nume)
            continue
        for url in linkuri:                            # încercăm oglinzile pe rând
            try:
                descarca(url, nume)
                break                                  # a mers -> nu mai încercăm altele
            except Exception as e:
                print("   nu merge:", type(e).__name__, "· încerc altă oglindă")
                if os.path.exists(nume):
                    os.remove(nume)
        else:                                          # (for...else) = nicio oglindă n-a mers
            raise RuntimeError(f"Nu am putut descărca {nume}. Arhiva trebuie pusă manual lângă notebook.")
    # rps.zip conține folderul rps/ (rock, paper, scissors), iar rps-test-set.zip folderul rps-test-set/
    dezarhiveaza("rps.zip", f"{DIR_BRUT}/tmp")
    dezarhiveaza("rps-test-set.zip", f"{DIR_BRUT}/tmp")
    shutil.move(f"{DIR_BRUT}/tmp/rps", f"{DIR_BRUT}/train")
    shutil.move(f"{DIR_BRUT}/tmp/rps-test-set", f"{DIR_BRUT}/val")
    shutil.rmtree(f"{DIR_BRUT}/tmp")

def adu_date():
    """Aduce pozele în funcție de SURSA și întoarce folderul în care se află."""
    if SURSA == "folder":                              # pozele sunt deja acolo
        return CALE_FOLDER
    if os.path.isdir(DIR_BRUT) and os.listdir(DIR_BRUT):   # la a doua rulare nu mai descărcăm
        print("✅ Pozele brute există deja în", DIR_BRUT)
        return DIR_BRUT
    if False:
        pass
    elif SURSA == "rps":
        adu_rps()
    elif SURSA == "drive":
        from google.colab import drive                 # disponibil doar în Google Colab
        drive.mount("/content/drive")
        if False:
            pass
        if not CALE_ZIP.endswith((".zip", ".tgz", ".tar.gz")) or not os.path.exists(CALE_ZIP):
            return CALE_FOLDER
        dezarhiveaza(CALE_ZIP, DIR_BRUT)
    elif SURSA == "zip":
        assert os.path.exists(CALE_ZIP), f"Nu găsesc arhiva {CALE_ZIP}."
        dezarhiveaza(CALE_ZIP, DIR_BRUT)
    elif SURSA == "url":
        nume = URL_ARHIVA.split("/")[-1]
        if not os.path.exists(nume):
            descarca(URL_ARHIVA, nume)
        dezarhiveaza(nume, DIR_BRUT)
    elif SURSA == "kaggle":
        # Kaggle cere o cheie API (kaggle.json). Dacă nu e pusă, ți-o cere ca upload.
        import subprocess, importlib.util, sys
        if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
            from google.colab import files
            print("Încarcă fișierul kaggle.json (Kaggle → Settings → API → Create New Token):")
            files.upload()
            os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
            shutil.move("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
            os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        if importlib.util.find_spec("kaggle") is None:      # se instalează doar dacă lipsește
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_SET, "--unzip", "-p", DIR_BRUT], check=True)
    else:
        raise ValueError(f"SURSA necunoscută: {SURSA}")
    return DIR_BRUT

def gaseste_radacina(cale):
    """Coboară prin foldere până găsește fie train/ + val/, fie folderele claselor."""
    while True:
        subfoldere = [d for d in sorted(os.listdir(cale))
                      if os.path.isdir(os.path.join(cale, d)) and not d.startswith((".", "__"))]
        if {"train", "val"} <= set(subfoldere):
            return cale, True
        are_poze = any(poze_din(os.path.join(cale, d)) for d in subfoldere)
        if are_poze or len(subfoldere) != 1:
            return cale, False
        cale = os.path.join(cale, subfoldere[0])

# Rulăm efectiv: aducem pozele și găsim de unde încep folderele cu clase
radacina, deja_impartit = gaseste_radacina(adu_date())
print("Rădăcina datelor:", radacina, "| deja împărțit în train/val:", deja_impartit)

### 3.2 Împărțirea în train și val
Fiecare poză ajunge ori în `train`, ori în `val`, niciodată în ambele. Când pozele nu sunt deja împărțite, le amestec cu un seed fix și pun 20% în validare. La piatră-hârtie-foarfecă împărțirea există deja, așa că aleg doar, la întâmplare, câte poze păstrez din fiecare clasă (`MAX_POZE_PE_CLASA`).

Dacă un set are poze aproape identice (de exemplu, mai multe poze făcute în rafală), ar trebui să stea toate în același set; altfel acuratețea pe validare iese prea mare.

In [ ]:
def pregateste_date(radacina, deja_impartit, destinatie):
    """Copiază pozele în destinatie/train/<clasa>/ și destinatie/val/<clasa>/. Întoarce lista claselor."""
    if os.path.isdir(destinatie):
        shutil.rmtree(destinatie)                      # refacem mereu, ca setările noi să aibă efect
    rng = random.Random(SEED)                          # generator aleator cu seed fix -> aceeași împărțire mereu
    if deja_impartit:                                  # clasele = subfolderele din train/
        surse = {imp: f"{radacina}/{imp}" for imp in ["train", "val"]}
        clase = sorted(d for d in os.listdir(surse["train"]) if os.path.isdir(f"{surse['train']}/{d}"))
    else:                                              # clasele = subfolderele care conțin poze
        clase = sorted(d for d in os.listdir(radacina)
                       if os.path.isdir(f"{radacina}/{d}") and poze_din(f"{radacina}/{d}"))
    if CLASE_ALESE:                                    # păstrăm doar clasele cerute în setări
        lipsa = set(CLASE_ALESE) - set(clase)
        assert not lipsa, f"Nu găsesc clasele {lipsa}. Clase disponibile: {clase}"
        clase = list(CLASE_ALESE)
    for clasa in clase:
        if deja_impartit:
            impartire = {imp: [f"{surse[imp]}/{clasa}/{f}" for f in poze_din(f"{surse[imp]}/{clasa}")]
                         for imp in ["train", "val"]}
            if MAX_POZE_PE_CLASA:                      # alegem la întâmplare o parte, păstrând proporția 80/20
                n_val = round(MAX_POZE_PE_CLASA * PROCENT_VAL)
                impartire = {"train": rng.sample(impartire["train"], min(len(impartire["train"]), MAX_POZE_PE_CLASA - n_val)),
                             "val": rng.sample(impartire["val"], min(len(impartire["val"]), n_val))}
        else:
            fisiere = [f"{radacina}/{clasa}/{f}" for f in poze_din(f"{radacina}/{clasa}")]
            rng.shuffle(fisiere)                       # amestecăm, apoi primele 20% merg la val
            if MAX_POZE_PE_CLASA:
                fisiere = fisiere[:MAX_POZE_PE_CLASA]
            n_val = max(1, round(len(fisiere) * PROCENT_VAL))   # cel puțin o poză de validare
            impartire = {"val": fisiere[:n_val], "train": fisiere[n_val:]}
        for imp, lista in impartire.items():           # copiem efectiv fișierele
            os.makedirs(f"{destinatie}/{imp}/{clasa}", exist_ok=True)
            for sursa in lista:
                shutil.copy(sursa, f"{destinatie}/{imp}/{clasa}/")
    return clase

CLASE = pregateste_date(radacina, deja_impartit, CALE_DATE)
print("Clase:", CLASE)
numara_poze(CALE_DATE)

### 3.3 Verificarea calității datelor

Verific trei lucruri: dacă se deschid toate pozele, dacă există poze identice în train și în val (pe care le scot din val) și dacă fiecare clasă are destule poze și clasele sunt echilibrate. Celula afișează un avertisment dacă ceva nu este în regulă.

In [ ]:
stricate = verifica_poze(CALE_DATE)            # mută deoparte pozele care nu se deschid
duplicate = gaseste_duplicate(CALE_DATE)        # caută poze identice în train și val

STERGE_DUPLICATE_DIN_VAL = True                 # True = le scot din val, ca validarea să fie corectă
if duplicate and STERGE_DUPLICATE_DIN_VAL:
    for _, din_val in duplicate:
        os.remove(f"{CALE_DATE}/{din_val}")
    print(f"🧹 Am șters {len(duplicate)} duplicate din val.")

tabel_poze = numara_poze(CALE_DATE)
tabel_poze["total"] = tabel_poze.sum(axis=1)   # coloană nouă: train + val
display(tabel_poze)

# Avertismente automate
if len(CLASE) < 2:
    print("❌ Ai nevoie de cel puțin 2 clase!")
for clasa, rand in tabel_poze.iterrows():
    if rand["total"] < 50:
        print(f"⚠️ Clasa '{clasa}' are doar {rand['total']} poze. Recomandat: cel puțin 50, ideal 100+.")
if tabel_poze["total"].max() > 2 * tabel_poze["total"].min():
    print("⚠️ Clasele sunt dezechilibrate (una are de peste 2x mai multe poze). Uită-te și la recall pe fiecare clasă.")

ax = numara_poze(CALE_DATE).plot.bar(rot=0, figsize=(7, 3.5), title="Câte poze are fiecare clasă")
ax.set_ylabel("poze"); ax.grid(axis="y", alpha=0.3); plt.show()

### 3.4 Explorare: cum arată pozele?
Afișez câte 6 poze din fiecare clasă și mărimea tipică a pozelor (mediana lățimii și a înălțimii).

In [ ]:
arata_mostre(CALE_DATE, CLASE, n=6)             # un rând de poze pentru fiecare clasă
dim = statistici_dimensiuni(CALE_DATE)          # lățimea și înălțimea fiecărei poze din train
display(dim.groupby("clasa")[["latime", "inaltime"]].median().rename(columns=lambda c: c + " (mediană)"))

**Ce urmăresc aici:** dacă pozele dintr-o clasă au ceva în comun care nu ține de gest (alt fundal, altă lumină, altă mărime). Dacă da, modelul ar putea învăța acel detaliu în loc de gest. În acest set toate pozele au fundal alb și aceeași mărime (300×300 px), deci singura diferență între clase este forma mâinii.

### 3.5 Transformări și augmentare
Folosesc aceleași augmentări ca în Varianta 1: decupare cu zoom, oglindire stânga-dreapta și schimbări mici de luminozitate și contrast. Pentru gesturi, toate sunt plauzibile: oglindirea transformă o mână dreaptă într-una stângă, iar gestul rămâne același. Pozele de antrenare au însă toate fundal alb, așa că pe poze reale, cu fundal încărcat, modelul poate avea probleme.

În celula următoare creez transformările, încarc pozele cu `ImageFolder` (eticheta este numele folderului), le grupez în loturi cu `DataLoader` și afișez o poză cu 5 variante augmentate ale ei.

In [ ]:
transform_train, transform_val = transformari_pentru(ARHITECTURA)
# Alte augmentări se pot defini aici, de exemplu:
# transform_train = transforms.Compose([
#     transforms.RandomResizedCrop(224), transforms.RandomRotation(15),
#     transforms.ToTensor(), transforms.Normalize(MEDIE, DEVIATIE)])

# ImageFolder: fiecare subfolder = o clasă; clasele primesc numere 0, 1, 2... în ordine alfabetică
date_train = datasets.ImageFolder(f"{CALE_DATE}/train", transform=transform_train)
date_val = datasets.ImageFolder(f"{CALE_DATE}/val", transform=transform_val)
CLASE = date_train.classes
print("clase:", date_train.class_to_idx, "| train:", len(date_train), "| val:", len(date_val))

# DataLoader: împarte pozele în loturi. La train le amestecăm (shuffle) la fiecare epocă; la validare nu e nevoie.
g = torch.Generator().manual_seed(SEED)
train_loader = torch.utils.data.DataLoader(date_train, batch_size=BATCH, shuffle=True,
                                           num_workers=NUM_WORKERS, generator=g)
val_loader = torch.utils.data.DataLoader(date_val, batch_size=32, num_workers=NUM_WORKERS)

# Prima poză din train: originalul + 5 variante augmentate (de fiecare dată altfel)
poza = Image.open(date_train.imgs[0][0]).convert("RGB")
fig, axe = plt.subplots(1, 6, figsize=(15, 3))
axe[0].imshow(poza); axe[0].set_title("originală", fontsize=9); axe[0].axis("off")
for ax in axe[1:]:
    arata_tensor(transform_train(poza), ax, "augmentată")
plt.show()

## 4 · Baseline: „ghicesc mereu clasa mare”
Cel mai simplu „model” nu se uită la poză și răspunde mereu cu clasa cea mai frecventă din train. Cu 3 clase egale, nimerește în aproximativ 33% din cazuri. Un model bun trebuie să-l depășească clar.

In [ ]:
nr_train = np.bincount(date_train.targets, minlength=len(CLASE))   # câte poze are fiecare clasă în train
clasa_mare = int(nr_train.argmax())                                  # clasa cu cele mai multe poze
y_val = np.array(date_val.targets)                                   # etichetele reale din validare
acc_baseline = (y_val == clasa_mare).mean()                          # cât de des ar nimeri „mereu clasa mare"
print("poze train pe clasă:", dict(zip(CLASE, nr_train.tolist())))
print(f"Baseline: răspunde mereu '{CLASE[clasa_mare]}' → acuratețe validare {acc_baseline:.1%}")
print(f"(Ghicitul la întâmplare ar da ~{1 / len(CLASE):.0%}.)")

## 5 · Modele
Compar două strategii, pornind de la ResNet18 pre-antrenat:
- **feature extraction**: tot modelul este înghețat și antrenez doar stratul nou de la final (`lr = 1e-3`);
- **fine-tuning**: dezgheț și ultimul bloc (`layer4`), cu o rată de învățare de 10 ori mai mică (`lr = 1e-4`).

Codul acceptă și `fine_tuning_complet` (tot modelul, `lr = 1e-5`), util doar când există multe poze.

Pentru fiecare strategie, `ruleaza_experiment` construiește un model nou, îl antrenează 5 epoci și afișează după fiecare epocă loss-ul și acuratețea pe train și pe validare.

In [ ]:
rezultate, istorice, modele = [], {}, {}       # aici strângem rezultatele fiecărui experiment
for strategie in STRATEGII:
    print(f"========== {ARHITECTURA} · {strategie} ==========")
    model, istoric, acc = ruleaza_experiment(ARHITECTURA, strategie, EPOCI, train_loader, val_loader, len(CLASE))
    modele[strategie], istorice[strategie] = model, istoric
    rezultate.append({"model": f"{ARHITECTURA} · {strategie}", "acuratețe validare": acc,
                      "parametri antrenați": sum(p.numel() for p in model.parameters() if p.requires_grad)})

Compar rezultatele: curbele de învățare, un tabel și un grafic cu acuratețea fiecărui model față de baseline. La final aleg automat modelul cu cea mai mare acuratețe pe validare.

In [ ]:
deseneaza_istoric(istorice)

# Tabel: baseline + fiecare experiment
comparatie = pd.DataFrame([{"model": "baseline (clasa mare)", "acuratețe validare": acc_baseline,
                            "parametri antrenați": 0}] + rezultate)
display(comparatie.style.format({"acuratețe validare": "{:.1%}", "parametri antrenați": "{:,}"}))

# Grafic cu bare orizontale
plt.figure(figsize=(7, 0.6 * len(comparatie) + 1.5))
bare = plt.barh(comparatie["model"], comparatie["acuratețe validare"], color="#4C78A8")
plt.bar_label(bare, labels=[f"{a:.1%}" for a in comparatie["acuratețe validare"]], padding=3)
plt.xlim(0, 1.08); plt.gca().invert_yaxis(); plt.xlabel("acuratețe validare"); plt.show()

# Alegem strategia cu acuratețea de validare cea mai mare
strategie_finala = max(rezultate, key=lambda r: r["acuratețe validare"])["model"].split(" · ")[1]
model_final = modele[strategie_finala]
print(f"🏆 Modelul ales: {ARHITECTURA} · {strategie_finala}")

## 6 · Evaluare: unde greșește modelul?
Acuratețea este o singură cifră. Ca să văd **unde** greșește modelul, mă uit la:
- **raportul de clasificare:** pentru fiecare clasă, **precizia** (din pozele numite „rock”, câte erau într-adevăr „rock”), **recall-ul** (din toate pozele „rock” reale, câte a recunoscut modelul) și **f1** (media armonică a celor două);
- **matricea de confuzie:** ce gesturi se confundă între ele.

In [ ]:
# Rulăm modelul ales pe toate pozele de validare
acc, y_true, y_pred, probs = evalueaza(model_final, val_loader)
print(f"Acuratețe validare: {acc:.1%}\n")
print(classification_report(y_true, y_pred, labels=range(len(CLASE)), target_names=CLASE, digits=3, zero_division=0))
matrice_confuzie(y_true, y_pred, CLASE, f"{ARHITECTURA} · {strategie_finala}")

Afișez apoi pozele greșite (deasupra fiecăreia: clasa reală, clasa prezisă și probabilitatea dată de model) și pozele la care modelul a ezitat cel mai mult.

In [ ]:
print("Greșelile, începând cu cele la care modelul era cel mai sigur pe el:")
fisiere_gresite = arata_greselile(date_val, y_true, y_pred, probs, CLASE)
print("Pozele la care modelul a ezitat cel mai mult:")
arata_nesigure(date_val, y_true, probs, CLASE)

## 7 · Demo: salvez modelul și îl folosesc pe poze noi
Salvez modelul într-un fișier `.pt` (greutăți, clase și arhitectură), apoi îl reîncarc, ca și cum l-aș folosi într-o aplicație separată, și fac o predicție pe o poză din validare.

In [ ]:
CALE_MODEL = f"{NUME_PROIECT}_model.pt"
salveaza_model(model_final, CLASE, ARHITECTURA, CALE_MODEL)

# ... mai târziu, în alt notebook sau într-o aplicație:
model_incarcat, clase_incarcate, arh = incarca_model(CALE_MODEL)
_, transform_predictie = transformari_pentru(arh)          # aceeași preprocesare ca la validare
print(prezice(date_val.imgs[0][0], model_incarcat, clase_incarcate, transform_predictie))

**Test pe mâini reale.** Modelul a văzut doar mâini desenate pe calculator, pe fundal alb. Ca să văd cum se descurcă în realitate, îl testez pe poze cu mâna mea, făcute cu telefonul: întâi pe fundal alb (o foaie de hârtie), apoi pe un fundal încărcat. Diferența dintre datele de antrenare și cele din realitate se numește **domain shift**. Modelul nu are o clasă „altceva”, așa că la predicțiile cu probabilitate sub 80% afișez „nu sunt sigur”.

In [ ]:
POZE_NOI = [
    # "https://....jpg",                      # link către o poză de pe internet
    # "/content/piatra_fundal_alb.jpg",      # poză proprie, făcută cu telefonul
    # "/content/foarfeca_fundal_incarcat.jpg",
]
PRAG_INCREDERE = 0.80   # sub prag afișez „nu sunt sigur” (modelul nu are o clasă „altceva”)
for sursa in POZE_NOI:
    try:
        clasa, incredere = prezice(sursa, model_incarcat, clase_incarcate, transform_predictie)
        print(f"{sursa} → {clasa} ({incredere:.0%})" + ("" if incredere >= PRAG_INCREDERE else "  ⚠️ nu sunt sigur"))
    except Exception as e:
        print("Nu am putut deschide", sursa, "|", e)
if not POZE_NOI:
    print("Lista POZE_NOI este goală.")

## 8 · Concluzii
Mai jos am adunat cifrele importante ale proiectului.

In [ ]:
print("REZUMATUL REZULTATELOR")
print(f"  • date: {len(date_train)} poze train, {len(date_val)} poze validare, {len(CLASE)} clase {CLASE}")
for _, r in comparatie.iterrows():
    print(f"  • {r['model']:<40} {r['acuratețe validare']:.1%}")
print(f"  • greșeli ale modelului ales: {int((y_true != y_pred).sum())} din {len(y_true)}")

**Limitări:**
- pozele de antrenare sunt generate pe calculator și au toate fundal alb, deci modelul poate avea probleme cu poze reale (domain shift);
- am folosit doar 150 de poze pe clasă și un singur seed;
- am ales modelul final tot pe validare, deci acuratețea obținută este puțin optimistă;
- modelul cunoaște doar cele 3 gesturi: pentru orice altă poză va răspunde tot cu unul dintre ele.

**Pași următori:** poze reale, cu fundaluri și lumini diferite; o clasă „altceva”; un set de test separat; alte arhitecturi (`efficientnet_b0`, `mobilenet_v3_large`).